In [1]:
import pickle
import os
import pandas as pd
from collections import Counter, defaultdict

In [2]:
res_discomat = pickle.load(open('../Matskraft_composition/code/inference/res_matskraft_composition_test.pkl', 'rb'))

# res_discomat = pickle.load(open('../data/res_discomat_just_expe.pkl', 'rb'))

In [3]:
print(res_discomat.keys())
print(res_discomat['pred_row_col_labels'][0])

dict_keys(['identifier', 'pred_table_type_labels', 'pred_row_col_labels', 'pred_edges', 'pred_tuples', '1_2_violations', '1_3_violations', '2_3_violations', '3_3_violations'])
{'row': [0, 0, 0, 0, 0, 0, 0], 'col': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [4]:
gold_disco_tuples_test = pickle.load(open('../Matskraft_composition/data/all_gold_tuples_test.pkl', 'rb'))
fpred_disco_tuples = res_discomat['pred_tuples']

In [5]:
# print(gold_disco_tuples_test[0:5])
# print()
# print(fpred_disco_tuples[0:5])

pred_disco_tuples = []
for tup in fpred_disco_tuples:
    if any(tup):
        pred_disco_tuples += tup
        
# print()
# print(pred_disco_tuples[0:5])

In [6]:
train_test_split = pickle.load(open('../Matskraft_composition/data/train_val_test_split.pkl', 'rb'))
make_pii_test = []
for (pii, tidx) in train_test_split['test']:
    make_pii_test.append(pii+'_'+str(tidx))
print(make_pii_test[:5])

['S0022309307010605_0', 'S0022309304006957_0', 'S0022309305003522_0', 'S0022309304003618_0', 'S0022309303004174_1']


In [7]:
pred_disco_tuples_test = []
for tup in pred_disco_tuples:
    if any(tup):
        #print(tup)
        uned_pii_tid = tup[0]
        parts = uned_pii_tid.split('_')
        pii_tid = '_'.join(parts[:2])
        #print(pii_tid)
        if pii_tid in make_pii_test:
            pred_disco_tuples_test.append(tup)

In [8]:
print(len(gold_disco_tuples_test), len(pred_disco_tuples_test))

9540 7161


In [9]:
def sort_comp(single_list_comp):
    '''
    for sorting  ordering of chemical components alphabetically
    '''
    single_list_comp_sorted = sorted(single_list_comp, key=lambda x: x[0])
    return single_list_comp_sorted

def get_composition_metrics_without_ids(gold_tuples, pred_tuples):
    gold_comps, pred_comps = defaultdict(set), defaultdict(set)
    
    for g in gold_tuples:
        underscore_count = g[0].count('_')
        if underscore_count == 5:
            new_item = '_'.join(g[0].split('_')[:-1])
        else:
            new_item = g[0]
        gold_comps[new_item].add((g[1], round(g[2], 2), g[3]))
#         print(gold_comps[g[0]])

    for p in pred_tuples:
        underscore_count = p[0].count('_')
        if underscore_count == 5:
            new_item = '_'.join(p[0].split('_')[:-1])
        else:
            new_item = p[0]
        pred_comps[new_item].add((p[1], round(p[2], 2), p[3]))
        
   
    for p, v in pred_comps.items():
        pred_comps[p] = set(sort_comp(list(v)))
        
    for g, v in gold_comps.items():
        gold_comps[g] = set(sort_comp(list(v)))

    prec = 0
    for p, v in pred_comps.items():
        if p in gold_comps and gold_comps[p] == v:
            prec += 1
    if len(pred_comps) > 0:
        prec /= len(pred_comps)
    else:
        prec = 0.0
    rec = 0
    for g, v in gold_comps.items():
        if g in pred_comps and pred_comps[g] == v:
            rec += 1
    rec /= len(gold_comps)
    fscore = 2 * prec * rec / (prec + rec) if (prec + rec > 0) else 0.0
    metrics = {'precision': prec, 'recall': rec, 'fscore': fscore}
    metrics = {m: round(v * 100, 2) for m, v in metrics.items()}
    return metrics

In [10]:
print(get_composition_metrics_without_ids(gold_disco_tuples_test, pred_disco_tuples_test))

{'precision': 82.31, 'recall': 62.97, 'fscore': 71.35}


In [11]:
material_gold, material_pred = {}, {}
for tup in gold_disco_tuples_test:
    key = tup[0]
    if key not in material_gold:
        material_gold[key] = []
    material_gold[key].append(tup)
    
for tup in pred_disco_tuples_test:
    key = tup[0]
    if key not in material_pred:
        material_pred[key] = []
    material_pred[key].append(tup)

In [12]:
#values_list = list(my_dict.values())
material_gold_list = list(material_gold.values())
material_pred_list = list(material_pred.values())
print(len(material_gold_list), len(material_pred_list))

2660 2035


In [13]:
print(material_gold_list[:5])
pickle.dump(material_pred_list, open('pred_materials.pkl', 'wb'))
pickle.dump(material_gold_list, open('gold_materials.pkl', 'wb'))

[[('S0022309304006957_0_2_1_0_1', 'PbO', 30.0, 'mol'), ('S0022309304006957_0_2_1_0_1', 'SiO2', 70.0, 'mol')], [('S0022309304006957_0_2_2_0_2', 'PbO', 33.0, 'mol'), ('S0022309304006957_0_2_2_0_2', 'SiO2', 67.0, 'mol')], [('S0022309304006957_0_2_3_0_3', 'PbO', 43.0, 'mol'), ('S0022309304006957_0_2_3_0_3', 'SiO2', 57.0, 'mol')], [('S0022309304006957_0_2_4_0_4', 'PbO', 50.0, 'mol'), ('S0022309304006957_0_2_4_0_4', 'SiO2', 50.0, 'mol')], [('S0022309304003618_0_1_1_0', 'Si', 0.336, 'mol'), ('S0022309304003618_0_1_1_0', 'Ca', 0.086, 'mol'), ('S0022309304003618_0_1_1_0', 'Na', 0.119, 'mol'), ('S0022309304003618_0_1_1_0', 'Co', 0.00071, 'mol'), ('S0022309304003618_0_1_1_0', 'O', 0.45829, 'mol')]]


In [14]:
pickle.dump(pred_disco_tuples_test, open('pred_disco_tuples_test.pkl', 'wb'))
pickle.dump(gold_disco_tuples_test, open('gold_disco_tuples_test.pkl', 'wb'))